# BAGIAN AI/MLP SPECIALIST

In [ ]:
# ==========================================
# 🛡️ SCRIPT ANTI-HILANG (JALANKAN PALING AWAL)
# ==========================================
from google.colab import drive
import os

# 1. Gembok Colab langsung ke Google Drive kamu
drive.mount('/content/drive')

# 2. Buat brankas khusus di Drive kamu
brankas_drive = '/content/drive/MyDrive/Proyek_ScamShield_Aman'
os.makedirs(brankas_drive, exist_ok=True)

print(f"\n✅ BRANKAS AKTIF! Semua kerja kerasmu akan otomatis diselamatkan ke: {brankas_drive}")

Mounted at /content/drive

✅ BRANKAS AKTIF! Semua kerja kerasmu akan otomatis diselamatkan ke: /content/drive/MyDrive/Proyek_ScamShield_Aman


In [ ]:
import pandas as pd
import os
from google.colab import drive

# Ensure Google Drive is mounted and brankas_drive is defined
# This makes the cell more robust even if previous cells are not run in order.
brankas_drive = '/content/drive/MyDrive/Proyek_ScamShield_Aman'
if not os.path.exists(brankas_drive):
    print("Warning: brankas_drive path does not exist. Attempting to mount Google Drive.")
    drive.mount('/content/drive')
    os.makedirs(brankas_drive, exist_ok=True)
    print(f"Brankas Drive re-checked/created at: {brankas_drive}")


# 1. Tentukan path file sesuai struktur di Colab
# Prioritize checking in brankas_drive and its common subfolders
possible_sms_paths = [
    os.path.join(brankas_drive, 'sms_spam_indo.csv'),
    os.path.join(brankas_drive, 'fakenewsindo', 'sms_spam_indo.csv'),
    os.path.join(brankas_drive, 'data', 'sms_spam_indo.csv'),
    os.path.join(brankas_drive, 'dataset', 'sms_spam_indo.csv'), # Another common subfolder
    os.path.join('/content/', 'sms_spam_indo.csv'), # Fallback to /content/
    os.path.join('/content/fakenewsindo/', 'sms_spam_indo.csv') # Original fallback from notebook
]

possible_hoax_paths = [
    os.path.join(brankas_drive, 'Data_latih.csv'),
    os.path.join(brankas_drive, 'smsspamindo', 'Data_latih.csv'),
    os.path.join(brankas_drive, 'data', 'Data_latih.csv'),
    os.path.join(brankas_drive, 'dataset', 'Data_latih.csv'), # Another common subfolder
    os.path.join('/content/', 'Data_latih.csv'), # Fallback to /content/
    os.path.join('/content/smsspamindo/', 'Data_latih.csv') # Original fallback from notebook
]

path_sms = None
for p in possible_sms_paths:
    if os.path.exists(p):
        path_sms = p
        break

path_hoax = None
for p in possible_hoax_paths:
    if os.path.exists(p):
        path_hoax = p
        break

if path_sms is None:
    raise FileNotFoundError(f"Could not find 'sms_spam_indo.csv' in any of the expected locations: {possible_sms_paths}. Please ensure the file is in your Google Drive 'Proyek_ScamShield_Aman' folder or a subfolder within it, or specify its correct path.")
if path_hoax is None:
    raise FileNotFoundError(f"Could not find 'Data_latih.csv' in any of the expected locations: {possible_hoax_paths}. Please ensure the file is in your Google Drive 'Proyek_ScamShield_Aman' folder or a subfolder within it, or specify its correct path.")


print(f"Membaca file SMS dari: {path_sms}")
print(f"Membaca file Hoax dari: {path_hoax}")

# 2. Memuat Dataset
df_sms = pd.read_csv(path_sms)
df_hoax = pd.read_csv(path_hoax)

# 3. Menyeragamkan Dataset SMS Spam
# Mengubah 'spam' menjadi 1 (Scam) dan 'ham' menjadi 0 (Aman)
df_sms['label'] = df_sms['Kategori'].map({'spam': 1, 'ham': 0})
df_sms = df_sms.rename(columns={'Pesan': 'teks'})
df_sms_clean = df_sms[['teks', 'label']].copy()

# 4. Menyeragamkan Dataset Hoax News
# Menggabungkan judul dan narasi agar konteks teks lebih kaya untuk IndoBERT
df_hoax['teks'] = df_hoax['judul'].fillna('') + ". " + df_hoax['narasi'].fillna('')
df_hoax_clean = df_hoax[['teks', 'label']].copy()

# 5. Meleburkan Kedua Dataset
df_final = pd.concat([df_sms_clean, df_hoax_clean], ignore_index=True)

# 6. Pembersihan Akhir (Drop nilai kosong dan duplikat)
df_final = df_final.dropna(subset=['teks', 'label'])
df_final = df_final.drop_duplicates(subset=['teks'])
df_final['label'] = df_final['label'].astype(int)

# 7. Menampilkan Hasil Konsolidasi Data
print("\n========================================")
print("✅ PROSES PENGGABUNGAN SELESAI!")
print("========================================")
print(f"Total baris data gabungan: {len(df_final)} baris")
print("\nDistribusi Kelas (0 = Aman, 1 = Scam/Hoax):")
print(df_final['label'].value_counts())

# Intip 5 data teratas
print("\nSampel data siap latih:")
display(df_final.head())

Membaca file SMS dari: /content/drive/MyDrive/Proyek_ScamShield_Aman/sms_spam_indo.csv
Membaca file Hoax dari: /content/drive/MyDrive/Proyek_ScamShield_Aman/Data_latih.csv

✅ PROSES PENGGABUNGAN SELESAI!
Total baris data gabungan: 5370 baris

Distribusi Kelas (0 = Aman, 1 = Scam/Hoax):
label
1    4037
0    1333
Name: count, dtype: int64

Sampel data siap latih:


,teks,label
0,Plg Yth: Simcard anda mendptkan bonus poin plu...,1
1,Iya ih ko sedih sih gtau kapan lg ke bandung :(,0
2,Kalau mau bikin model/controller mending per a...,0
3,Selamat nama1. Semoga selalu menempuh hidup ya...,0
4,Tingkatkan nilai isi ulang Anda selanjutnya mi...,1


In [ ]:
# Simpan data bersih agar tidak hilang jika Colab ter-restart
df_final.to_csv('dataset_scamshield_clean.csv', index=False)
print("Data berhasil disimpan ke 'dataset_scamshield_clean.csv'")

# Instal library wajib untuk NLP
!pip install transformers datasets accelerate scikit-learn

Data berhasil disimpan ke 'dataset_scamshield_clean.csv'


In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import torch

# 1. Bagi data: 80% untuk belajar (Train), 20% untuk ujian (Test)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_final['teks'].tolist(),
    df_final['label'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Jumlah data latih: {len(train_texts)}")
print(f"Jumlah data uji: {len(test_texts)}")

# 2. Panggil Tokenizer asli IndoBERT
print("\nMengunduh Tokenizer IndoBERT...")
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

# 3. Proses Tokenisasi
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

# 4. Ubah ke format Dataset PyTorch
class ScamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ScamDataset(train_encodings, train_labels)
test_dataset = ScamDataset(test_encodings, test_labels)

print("✅ Data berhasil di-tokenisasi dan siap dimasukkan ke dalam model!")

Jumlah data latih: 4296
Jumlah data uji: 1074

Mengunduh Tokenizer IndoBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Data berhasil di-tokenisasi dan siap dimasukkan ke dalam model!


In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Fungsi untuk menghitung nilai rapot AI (Akurasi, F1-Score)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Definisikan mapping label kita
id2label = {0: "NOT_SCAM", 1: "SCAM"}
label2id = {"NOT_SCAM": 0, "SCAM": 1}

# 2. Unduh Otak Utama IndoBERT
print("Mengunduh Model IndoBERT...")
model = AutoModelForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p1",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# ========================================================
# 3. PENGATURAN PROSES BELAJAR (VERSI ANTI-HILANG KE DRIVE)
# ========================================================
brankas_drive = '/content/drive/MyDrive/Proyek_ScamShield_Aman'

training_args = TrainingArguments(
    output_dir=f'{brankas_drive}/checkpoint_scamshield', # <--- Checkpoint langsung masuk Drive
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=f'{brankas_drive}/logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",                 # AI menyimpan dirinya otomatis tiap 1 putaran selesai
    load_best_model_at_end=True,
    save_total_limit=2                     # Hemat memori Drive, cukup simpan 2 checkpoint terakhir
)

# 4. Siapkan Guru (Trainer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# 5. GAS BELAJAR!
print("🚀 Memulai proses training AI... (Tunggu ya, ini butuh waktu beberapa menit)")
trainer.train()

# 6. Simpan Model yang sudah pintar LANGSUNG KE DRIVE
print("\n✅ Training selesai! Menyimpan model FINAL ke Google Drive...")
model.save_pretrained(f"{brankas_drive}/scamshield_model_final")
tokenizer.save_pretrained(f"{brankas_drive}/scamshield_model_final")
print("🎉 AMAN 100%! Model SCAMSHIELD berhasil disimpan permanen di Google Drive!")

Mengunduh Model IndoBERT...


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Training selesai! Menyimpan model FINAL ke Google Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 AMAN 100%! Model SCAMSHIELD berhasil disimpan permanen di Google Drive!


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/Proyek_ScamShield_Aman"))

['Data_latih.csv', 'sms_spam_indo.csv', 'scamshield_model_final', 'sesi_scamshield.session', 'dataset_scamshield_clean.csv', 'checkpoint_scamshield']


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Load Model & Tokenizer dari folder lokal hasil belajarmu
print("⏳ Memuat otak AI ScamShield dari penyimpanan lokal...")
model_path = "/content/drive/MyDrive/Proyek_ScamShield_Aman/scamshield_model_final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# 2. Buat Fungsi Pengecekan Cepat (Ini yang nanti dikasih ke Data Engineer)
def cek_pesan_telegram(teks):
    # Ubah teks jadi angka agar dimengerti AI
    inputs = tokenizer(teks, return_tensors="pt", truncation=True, padding=True, max_length=128)

    # AI menebak hasil (tanpa menghitung gradient agar hemat memori)
    with torch.no_grad():
        outputs = model(**inputs)

    # Hitung probabilitas (Confidence Score)
    probs = F.softmax(outputs.logits, dim=1).squeeze()
    pred_id = torch.argmax(probs).item()
    confidence = probs[pred_id].item() * 100

    # Terjemahkan hasil ke bahasa manusia
    label_map = {0: "✅ AMAN (NOT_SCAM)", 1: "🚨 PENIPUAN (SCAM)"}
    hasil = label_map[pred_id]

    print(f"Pesan Masuk : \"{teks}\"")
    print(f"Status      : {hasil}")
    print(f"Confidence  : {confidence:.2f}%\n")
    print("=" * 60)

# =========================================================
# 3. AREA UJI COBA (Silakan ganti teksnya sesuka hatimu!)
# =========================================================
print("✅ Sistem Siap! Memulai Simulasi Pengecekan...\n")
print("=" * 60)

# Ujian 1: Obrolan Normal (Sesuai dataset Kaggle)
cek_pesan_telegram("Eh besok jadi kumpul bahas tugas Big Data di perpus jam 10 pagi ga? Jangan pada telat ya, gue udah booking tempat.")

# Ujian 2: Penipuan Kekinian (Harusnya SCAM)
cek_pesan_telegram("Kerja santai dari rumah modal HP! Cukup bantu like dan subscribe Youtube dibayar 500rb - 2jt per hari. Segera chat admin WA 0812345xxx untuk daftar dan bayar biaya registrasi 50rb.")

# Ujian 3: Hoax/Scam Klasik (Harusnya SCAM)
cek_pesan_telegram("PESAN RESMI: Selamat nomor anda memenangkan undian 100 juta dari Shopee. Silakan klik link berikut untuk klaim pencairan dana.")

⏳ Memuat otak AI ScamShield dari penyimpanan lokal...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Sistem Siap! Memulai Simulasi Pengecekan...

Pesan Masuk : "Eh besok jadi kumpul bahas tugas Big Data di perpus jam 10 pagi ga? Jangan pada telat ya, gue udah booking tempat."
Status      : ✅ AMAN (NOT_SCAM)
Confidence  : 99.46%

Pesan Masuk : "Kerja santai dari rumah modal HP! Cukup bantu like dan subscribe Youtube dibayar 500rb - 2jt per hari. Segera chat admin WA 0812345xxx untuk daftar dan bayar biaya registrasi 50rb."
Status      : 🚨 PENIPUAN (SCAM)
Confidence  : 96.68%

Pesan Masuk : "PESAN RESMI: Selamat nomor anda memenangkan undian 100 juta dari Shopee. Silakan klik link berikut untuk klaim pencairan dana."
Status      : 🚨 PENIPUAN (SCAM)
Confidence  : 97.07%



# BAGIAN SKRIP DATA ENGINEER

In [ ]:
!sudo apt install tesseract-ocr
!pip install pytesseract pillow telethon

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.8/789.8 kB 27.6 MB/s eta 0:00:00
  Created wheel for pyaes: filename=pyaes-1.6.1-py3-none-any.whl size=26347 sha256=3e0922cf310d6fb11d044d85aca120dc2ff38d3a1fa9e5a6d700c794785baccc
  Stored in directory: /root/.cache/pip/wheels/d9/43/32/ec313dd557e88e419489b9df40c04dad7b99576bde162496f8
Successfully built pyaes


In [ ]:
import pytesseract
from PIL import Image

# Fungsi untuk membaca teks dari gambar
def baca_teks_dari_gambar(image_path):
    print(f"🔍 [OCR] Mengekstrak teks dari gambar: {image_path}...")
    try:
        # Buka file gambar
        img = Image.open(image_path)
        # Ekstrak teks menggunakan Tesseract
        teks_hasil = pytesseract.image_to_string(img)

        # Bersihkan spasi atau baris kosong yang berlebihan
        teks_bersih = " ".join(teks_hasil.split())

        if teks_bersih:
            print(f"📝 Hasil Ekstraksi: \"{teks_bersih}\"")
        else:
            print("📝 Hasil Ekstraksi: (Tidak ada teks yang terdeteksi)")

        return teks_bersih
    except Exception as e:
        print(f"❌ Gagal membaca gambar: {e}")
        return ""

# ==========================================
# CARA TEST-NYA (Bisa langsung di Colab)
# ==========================================
# 1. Upload gambar poster loker ke Colab (misal namanya: poster.jpg)
# 2. Panggil fungsi OCR
# teks_dari_poster = baca_teks_dari_gambar("poster.jpg")

# 3. Lempar hasilnya ke AI kamu!
# if teks_dari_poster:
#     cek_pesan_telegram(teks_dari_poster)

In [ ]:
import pytesseract
from PIL import Image
from telethon import TelegramClient, events
import os
import asyncio
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Define brankas_drive for this cell to ensure it's self-contained
brankas_drive = '/content/drive/MyDrive/Proyek_ScamShield_Aman'

# =========================================================
# 1. ATUR DEVICE & LOAD MODEL KE GPU
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Sistem berjalan di: {device.type.upper()}")

print("⏳ Memuat otak AI ScamShield dari Google Drive...")
model_path = f'{brankas_drive}/scamshield_model_final'

try:
    ai_tokenizer = AutoTokenizer.from_pretrained(model_path)
    ai_model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    print("✅ Model AI Berhasil Dimuat di GPU!")
except Exception as e:
    print(f"❌ Gagal memuat model. Error: {e}")

# =========================================================
# 2. FUNGSI CEK AI (SUDAH FIXED DEVICE MISMATCH)
# =========================================================
def cek_pesan_telegram(teks):
    if not teks.strip():
        return "Teks Kosong", 0.0

    # Tokenisasi data
    inputs = ai_tokenizer(teks, return_tensors="pt", truncation=True, padding=True, max_length=128)

    # 🔥 FIX: Pindahkan seluruh input object ke GPU
    inputs = inputs.to(device)

    # Prediksi menggunakan AI
    with torch.no_grad():
        outputs = ai_model(**inputs)

    probs = F.softmax(outputs.logits, dim=1).squeeze()

    if probs.dim() == 0:
        pred_id = torch.argmax(outputs.logits).item()
        confidence = 100.0
    else:
        pred_id = torch.argmax(probs).item()
        confidence = probs[pred_id].item() * 100

    label_map = {0: "✅ AMAN (NOT_SCAM)", 1: "🚨 PENIPUAN (SCAM)"}
    return label_map[pred_id], confidence

# =========================================================
# 3. FUNGSI OCR LOKAL
# =========================================================
def baca_teks_dari_gambar(image_path):
    print(f"🔍 [OCR] Mengekstrak teks dari gambar: {image_path}...")
    try:
        img = Image.open(image_path)
        teks_hasil = pytesseract.image_to_string(img)
        teks_bersih = " ".join(teks_hasil.split())
        return teks_bersih
    except Exception as e:
        print(f"❌ Gagal membaca gambar via OCR: {e}")
        return ""

# =========================================================
# 4. KONFIGURASI TELEGRAM CLIENT
# =========================================================
load_dotenv()
api_id = os.getenv("TELEGRAM_API_ID")
api_hash = os.getenv("TELEGRAM_API_HASH")

session_path = f'{brankas_drive}/sesi_scamshield'
client = TelegramClient(session_path, api_id, api_hash)

# Menggunakan ID Numerik Permanen agar kebal dari perubahan nama/error teks
grup_target = [
    -1001727603389,  # INFO LOKER CIKARANG 2026
    -1001075178282,  # INFO LOKER KARAWANG 2026
    -1001296845898,  # INFO LOKER BOGOR 2026
    -1001296711022,  # LOWONGAN INDONESIA🇲🇨
    -1001062241629,  # Lowker Migas
    -1001390228011,  # Lowongan Kerja Jakarta
    -1001143132643,  # INFO LOWONGAN KERJA
    -1001103714494,  # LOKER BUMN 45
    -1001453343331,  # Lowongan Kerja Terbaru
    -1001157527957,  # LOWONGAN KERJA BATAM CITY
    -1001160377850,  # INFO LOKER INDONESIA OFFICIAL
    -1001357623702,  # DISNAKERJA Official
    -1001278399434,  # Lowongan Kerja
    -1001479484104,  # LOWONGANKERJA
    -1001308205992,  # Karirpedia
    -1001125357217,  # Lowongan Kerja Indonesia
    -1001230189353,  # Loker Fresh Graduation
    -1001387167231,  # LOKER SURABAYA KARIR
    -1001808396515,  # 🔥 LoKer 🔥
    -1001241553199,  # Indonesian Freelancer
    -1001688543278,  # Informasi Loker — Dikelola Worksfess
    -1001496896117,  # LOKERMAGANG

    # Anda juga tetap bisa menggabungkannya dengan username seperti biasa:
    '@openkerja',
    '@lowkerLokalIndo',
    '@lowongancpnsbumn'
]

# =========================================================
# 5. HANDLER RADAR REAL-TIME
# =========================================================
@client.on(events.NewMessage(chats=grup_target))
async def handler_pesan_baru(event):
    print("\n" + "=" * 60)

    # Ambil Info Pengirim & Grup
    nama_pengirim = "Unknown"
    try:
        pengirim = await event.get_sender()
        if pengirim and hasattr(pengirim, 'username') and pengirim.username:
            nama_pengirim = pengirim.username
        elif pengirim and hasattr(pengirim, 'first_name') and pengirim.first_name:
            nama_pengirim = pengirim.first_name
    except:
        pass

    try:
        chat = await event.get_chat()
        nama_grup = chat.title if hasattr(chat, 'title') else chat.username
    except:
        nama_grup = "Grup Tidak Diketahui"

    print(f"📥 [PESAN BARU] Dari: @{nama_pengirim} di [{nama_grup}]")
    teks_pesan = event.raw_text or ""

    # Jika ada lampiran gambar
    if event.photo:
        print("📸 Mendeteksi gambar! Mengunduh lampiran untuk OCR...")
        path_gambar = f"temp_loker_{event.id}.jpg"
        await event.download_media(file=path_gambar)

        teks_pesan = baca_teks_dari_gambar(path_gambar)

        if os.path.exists(path_gambar):
            os.remove(path_gambar)

    # Kirim ke AI jika ada teks yang siap dianalisis
    if teks_pesan.strip():

        # 🔥 TAMBAHAN UNTUK MENAMPILKAN ISI PESAN 🔥
        print("-" * 40)
        print("📄 ISI PESAN:")
        # Menampilkan max 400 karakter agar log rapi (AI tetap memproses teks penuh)
        if len(teks_pesan) > 400:
            print(f"{teks_pesan[:400]}\n... [TEKS TERLALU PANJANG, DIPOTONG]")
        else:
            print(teks_pesan)
        print("-" * 40)

        print("🤖 Menyerahkan teks ke AI IndoBERT...")
        try:
            status_ai, confidence_ai = cek_pesan_telegram(teks_pesan)
            print(f"📊 HASIL ANALISIS RADAR SCAMSHIELD:")
            print(f"   > Status Hasil : {status_ai}")
            print(f"   > Akurasi AI   : {confidence_ai:.2f}%")
        except Exception as ai_error:
            print(f"❌ AI gagal memproses teks: {ai_error}")
    else:
        print("📝 Hasil Ekstraksi: (Tidak ada teks/gambar yang valid untuk dianalisis)")

    print("=" * 60)

# =========================================================
# 6. RUN SYSTEM
# =========================================================
print("📡 Radar SCAMSHIELD V2 Aktif Penuh!")
await client.start()

# 🔥 FIX ENTITY ERROR: Paksa Telethon mengingat semua daftar grup di HP Anda
print("🔄 Mensinkronisasi ingatan daftar grup Telegram...")
await client.get_dialogs()
print("✅ Sinkronisasi berhasil! Radar siap menangkap pesan.")

await client.run_until_disconnected()

🖥️ Sistem berjalan di: CUDA
⏳ Memuat otak AI ScamShield dari Google Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model AI Berhasil Dimuat di GPU!
📡 Radar SCAMSHIELD V2 Aktif Penuh!
🔄 Mensinkronisasi ingatan daftar grup Telegram...
✅ Sinkronisasi berhasil! Radar siap menangkap pesan.


📥 [PESAN BARU] Dari: @CIIO di [INFO LOKER KARAWANG 2026]
----------------------------------------
📄 ISI PESAN:
Hii
----------------------------------------
🤖 Menyerahkan teks ke AI IndoBERT...
📊 HASIL ANALISIS RADAR SCAMSHIELD:
   > Status Hasil : ✅ AMAN (NOT_SCAM)
   > Akurasi AI   : 98.43%
📥 [PESAN BARU] Dari: @Unknown di [INFO LOKER KARAWANG 2026]
----------------------------------------
📄 ISI PESAN:
Hii
----------------------------------------
🤖 Menyerahkan teks ke AI IndoBERT...
📊 HASIL ANALISIS RADAR SCAMSHIELD:
   > Status Hasil : ✅ AMAN (NOT_SCAM)
   > Akurasi AI   : 98.43%

📥 [PESAN BARU] Dari: @CIIO di [INFO LOKER KARAWANG 2026]
----------------------------------------
📄 ISI PESAN:
Selamat pagii
----------------------------------------
🤖 Menyerahkan teks ke AI IndoBERT...
📊 HASIL ANALISIS RADAR SCAMSHIEL

CancelledError: 